In [38]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

In [39]:
class AdaBoostScratch:
    def __init__(self, n_estimators=50):
        self.n_estimators = n_estimators
        self.alphas = []
        self.models = []

    def fit(self, X, y):
        n_samples = X.shape[0]
        y = np.where(y <= 0, -1, 1)
        w = np.full(n_samples, 1 / n_samples)

        for t in range(self.n_estimators):
            stump = DecisionTreeClassifier(max_depth=1)
            stump.fit(X, y, sample_weight=w)
            pred = stump.predict(X)

            miss = (pred != y).astype(int)
            err = np.sum(w * miss) / np.sum(w)
            err = np.clip(err, 1e-10, 1 - 1e-10)

            alpha = 0.5 * np.log((1 - err) / err)

            w = w * np.exp(-alpha * y * pred)
            w = w / np.sum(w)

            self.alphas.append(alpha)
            self.models.append(stump)

            # resampling step
            cum_w = np.cumsum(w)
            new_indices = []
            for _ in range(n_samples):
                r = np.random.uniform(0, 1)
                idx = np.searchsorted(cum_w, r)
                new_indices.append(idx)
            X, y = X[new_indices], y[new_indices]
            w = np.full(n_samples, 1 / n_samples)

    # ⚠️ yeh method 'fit' ke BAHAR, class ke ANDAR hona chahiye (same indentation level as fit/__init__)
    def predict(self, X):
        clf_preds = np.array([alpha * model.predict(X) 
                               for alpha, model in zip(self.alphas, self.models)])
        final_pred = np.sign(np.sum(clf_preds, axis=0))
        return final_pred

In [40]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [41]:
X, y = make_classification(n_samples=500, n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [42]:
model = AdaBoostScratch(n_estimators=50)
model.fit(X_train, y_train)

In [43]:
preds = model.predict(X_test)
y_test_signed = np.where(y_test <= 0, -1, 1)

print("Scratch Accuracy:", accuracy_score(y_test_signed, preds))

Scratch Accuracy: 0.73
